In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FlightDelayPredictionBigData") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "100s") \
    .config("spark.rdd.compress", "true") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

HDFS_PATH = "hdfs://localhost:9000/data/"

In [2]:
airlines = spark.read.csv(HDFS_PATH + "airlines.csv", header=True, inferSchema=True, sep = ",")
airlines.show(2)

+---------+--------------------+
|IATA_CODE|             AIRLINE|
+---------+--------------------+
|       UA|United Air Lines ...|
|       AA|American Airlines...|
+---------+--------------------+
only showing top 2 rows


In [3]:
airports = spark.read.csv(HDFS_PATH + "airports.csv", header=True, inferSchema=True, sep = ",")
airports.show(2)

+---------+--------------------+---------+-----+-------+--------+---------+
|IATA_CODE|             AIRPORT|     CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+--------------------+---------+-----+-------+--------+---------+
|      ABE|Lehigh Valley Int...|Allentown|   PA|    USA|40.65236| -75.4404|
|      ABI|Abilene Regional ...|  Abilene|   TX|    USA|32.41132| -99.6819|
+---------+--------------------+---------+-----+-------+--------+---------+
only showing top 2 rows


In [4]:
flights = spark.read.csv(HDFS_PATH + "flights.csv", header=True, inferSchema=True, sep = ",")
flights.show(2)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

In [5]:
from pyspark.sql.functions import col, when, lpad, concat, lit, substring

TIME_COLS = ['SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME']

for tcol in TIME_COLS:
    time_string_col = lpad(
        when(col(tcol) == 2400, 0)
        .otherwise(col(tcol))
        .cast("int").cast("string"),
        4, "0"
    )

    flights = flights.withColumn(
        tcol,
        concat(substring(time_string_col, 1, 2), lit(":"), substring(time_string_col, 3, 2))
    )


In [6]:
from pyspark.sql import functions as F

flights = flights.filter(F.col("CANCELLED") == 0)

flights = flights.withColumn(
    "label",
    F.when(F.col("ARRIVAL_DELAY") > 15, 1).otherwise(0)
)

In [7]:
flights = flights.withColumn(
    "DEP_TIME_MINUTES",
    F.split(F.col("DEPARTURE_TIME"), ":")[0].cast("int") * 60 +
    F.split(F.col("DEPARTURE_TIME"), ":")[1].cast("int")
)

In [8]:
flights.show(2)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+-----+----------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|label|DEP_TIME_MINUTES|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+---

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DEP_TIME_MINUTES", "DISTANCE", "DEPARTURE_DELAY"]

stages = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages.append(assembler)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
    # maxIter=20,
    # regParam=0.1,
    # elasticNetParam=0.0
)
stages.append(lr)

pipeline = Pipeline(stages=stages)

In [10]:
train_data, test_data = flights.randomSplit([0.8, 0.2], seed=42)

total_count = train_data.count()
delay_count = train_data.filter(F.col("label") == 1.0).count()
on_time_count = total_count - delay_count

weight_for_delay = total_count / (2.0 * delay_count)
weight_for_ontime = total_count / (2.0 * on_time_count)

train_data_weighted = train_data.withColumn(
    "classWeight",
    F.when(F.col("label") == 1.0, weight_for_delay).otherwise(weight_for_ontime)
)

# ── OPTIMIZATION: Cache & Persist ──────────────────────────────
print("Caching flights dataset...")
flights.cache()

print("Persisting train/test splits...")
train_data.persist()
test_data.persist()
train_data_weighted.persist()

print(f"Train size: {train_data.count():,}")
print(f"Test size:  {test_data.count():,}")
print("Cache & Persist completed.")

model = pipeline.fit(train_data_weighted)

predictions = model.transform(test_data)

Caching flights dataset...
Persisting train/test splits...
Train size: 4,583,896
Test size:  1,145,299
Cache & Persist completed.


In [11]:
model_path = "model/flight_logistic_model"
model.write().overwrite().save(model_path)

In [12]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9363
2. Accuracy:      0.9075
3. Precision:     0.9156
4. Recall:        0.9075
5. F1-Score:      0.9103


In [13]:
from sklearn.metrics import classification_report
y_compare = predictions.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare["label"], y_compare["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.92      0.94    941207
       Delay       0.70      0.83      0.76    204092

    accuracy                           0.91   1145299
   macro avg       0.83      0.88      0.85   1145299
weighted avg       0.92      0.91      0.91   1145299



In [14]:
import pandas as pd
import numpy as np

lr_model = model.stages[-1]
coefficients = lr_model.coefficients.toArray()

features_metadata = predictions.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===
                     Feature_Name  Coefficient  Absolute_Importance
0   DESTINATION_AIRPORT_Vec_13459   -10.529381            10.529381
1   DESTINATION_AIRPORT_Vec_11503   -10.112292            10.112292
2   DESTINATION_AIRPORT_Vec_13541   -10.060583            10.060583
3   DESTINATION_AIRPORT_Vec_10666    -9.656471             9.656471
4        ORIGIN_AIRPORT_Vec_14960    -9.333105             9.333105
5        ORIGIN_AIRPORT_Vec_14006    -9.129696             9.129696
6   DESTINATION_AIRPORT_Vec_12016    -9.129185             9.129185
7   DESTINATION_AIRPORT_Vec_13127    -9.093438             9.093438
8        ORIGIN_AIRPORT_Vec_10268    -8.919617             8.919617
9        ORIGIN_AIRPORT_Vec_13502    -8.560259             8.560259
10       ORIGIN_AIRPORT_Vec_11503    -8.373560             8.373560
11       ORIGIN_AIRPORT_Vec_14150    -6.235228             6.235228
12  DESTINATION_AIRPORT_Vec_11315    -4.791955         

In [15]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DEP_TIME_MINUTES", "DISTANCE", "DEPARTURE_DELAY"]

stages_dt = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_dt += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_dt = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_dt.append(assembler_dt)

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",   # thêm
    maxDepth=5,
    maxBins=32,
    impurity="gini"
)
stages_dt.append(dt)

pipeline_dt = Pipeline(stages=stages_dt)

In [16]:
model_dt = pipeline_dt.fit(train_data_weighted)

predictions_dt = model_dt.transform(test_data)

In [17]:
model_dt_path = "model/flight_decision_tree_model"
model_dt.write().overwrite().save(model_dt_path)

In [18]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_dt)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_dt)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_dt)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_dt)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_dt)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.8457
2. Accuracy:      0.9110
3. Precision:     0.9169
4. Recall:        0.9110
5. F1-Score:      0.9132


In [19]:
y_compare_dt = predictions_dt.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_dt["label"], y_compare_dt["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.93      0.95    941207
       Delay       0.72      0.82      0.77    204092

    accuracy                           0.91   1145299
   macro avg       0.84      0.88      0.86   1145299
weighted avg       0.92      0.91      0.91   1145299



In [20]:
dt_model = model_dt.stages[-1]
feature_importances = dt_model.featureImportances.toArray()

features_metadata = predictions_dt.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (DECISION TREE) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (DECISION TREE) ===
                     Feature_Name  Importance
0                 DEPARTURE_DELAY    0.998889
1                  AIRLINE_Vec_WN    0.000667
2                  AIRLINE_Vec_UA    0.000361
3     DESTINATION_AIRPORT_Vec_LAX    0.000083
4   DESTINATION_AIRPORT_Vec_12264    0.000000
5     DESTINATION_AIRPORT_Vec_GNV    0.000000
6     DESTINATION_AIRPORT_Vec_ISN    0.000000
7   DESTINATION_AIRPORT_Vec_14492    0.000000
8   DESTINATION_AIRPORT_Vec_14683    0.000000
9     DESTINATION_AIRPORT_Vec_BFL    0.000000
10    DESTINATION_AIRPORT_Vec_LNK    0.000000
11    DESTINATION_AIRPORT_Vec_BMI    0.000000
12    DESTINATION_AIRPORT_Vec_TVC    0.000000
13    DESTINATION_AIRPORT_Vec_BTV    0.000000
14    DESTINATION_AIRPORT_Vec_MFR    0.000000


In [21]:
from pyspark.ml.classification import GBTClassifier

stages_gbt = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_gbt += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_gbt = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_gbt.append(assembler_gbt)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",   # thêm
    maxIter=10,
    maxDepth=4,
    stepSize=0.1
)
stages_gbt.append(gbt)

pipeline_gbt = Pipeline(stages=stages_gbt)

In [22]:
model_gbt = pipeline_gbt.fit(train_data_weighted)

predictions_gbt = model_gbt.transform(test_data)

In [23]:
model_gbt_path = "model/flight_gbt_model"
model_gbt.write().overwrite().save(model_gbt_path)

In [24]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_gbt)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_gbt)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_gbt)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_gbt)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_gbt)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9325
2. Accuracy:      0.9097
3. Precision:     0.9162
4. Recall:        0.9097
5. F1-Score:      0.9121


In [25]:
y_compare_gbt = predictions_gbt.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_gbt["label"], y_compare_gbt["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.93      0.94    941207
       Delay       0.71      0.82      0.76    204092

    accuracy                           0.91   1145299
   macro avg       0.84      0.88      0.85   1145299
weighted avg       0.92      0.91      0.91   1145299



In [26]:
gbt_model = model_gbt.stages[-1]
feature_importances = gbt_model.featureImportances.toArray()

features_metadata = predictions_gbt.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (GBT) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (GBT) ===
                   Feature_Name  Importance
0               DEPARTURE_DELAY    0.943420
1                AIRLINE_Vec_DL    0.012996
2                AIRLINE_Vec_WN    0.011946
3                AIRLINE_Vec_UA    0.009778
4                      DISTANCE    0.009657
5   DESTINATION_AIRPORT_Vec_LAX    0.002410
6   DESTINATION_AIRPORT_Vec_ORD    0.002277
7                   MONTH_Vec_2    0.001486
8                AIRLINE_Vec_US    0.001341
9        ORIGIN_AIRPORT_Vec_ATL    0.001224
10             DEP_TIME_MINUTES    0.000758
11               AIRLINE_Vec_HA    0.000605
12       ORIGIN_AIRPORT_Vec_LGA    0.000577
13                  MONTH_Vec_1    0.000555
14  DESTINATION_AIRPORT_Vec_HNL    0.000496


In [27]:
# Tính ratio
total = train_data.count()
n_delay = train_data.filter(F.col("label") == 1).count()
n_ontime = train_data.filter(F.col("label") == 0).count()

weight_delay  = total / (2 * n_delay)
weight_ontime = total / (2 * n_ontime)

train_data_w = train_data.withColumn(
    "classWeight",
    F.when(F.col("label") == 1, weight_delay).otherwise(weight_ontime)
)

In [28]:
from pyspark.ml.classification import RandomForestClassifier

stages_rf = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_rf += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_rf = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_rf.append(assembler_rf)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",   # thêm dòng này
    numTrees=100,
    maxDepth=5,
    seed=42
)
stages_rf.append(rf)

pipeline_rf = Pipeline(stages=stages_rf)

In [29]:
model_rf = pipeline_rf.fit(train_data_weighted)  # dùng train_data_w thay vì train_data
predictions_rf = model_rf.transform(test_data)

In [30]:
model_rf_path = "model/flight_random_forest_model"
model_rf.write().overwrite().save(model_rf_path)

In [31]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_rf)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_rf)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_rf)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_rf)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_rf)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.8894
2. Accuracy:      0.9127
3. Precision:     0.9162
4. Recall:        0.9127
5. F1-Score:      0.9141


In [32]:
y_compare_rf = predictions_rf.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_rf["label"], y_compare_rf["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.94      0.95    941207
       Delay       0.73      0.80      0.77    204092

    accuracy                           0.91   1145299
   macro avg       0.85      0.87      0.86   1145299
weighted avg       0.92      0.91      0.91   1145299



In [33]:
rf_model = model_rf.stages[-1]
feature_importances = rf_model.featureImportances.toArray()

features_metadata = predictions_rf.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (RANDOM FOREST) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (RANDOM FOREST) ===
                     Feature_Name  Importance
0                 DEPARTURE_DELAY    0.208203
1                DEP_TIME_MINUTES    0.142107
2                  AIRLINE_Vec_DL    0.063441
3                  AIRLINE_Vec_NK    0.060582
4                     MONTH_Vec_2    0.042448
5                     MONTH_Vec_6    0.032774
6          ORIGIN_AIRPORT_Vec_ORD    0.029282
7                     MONTH_Vec_9    0.028957
8                    MONTH_Vec_11    0.024674
9               DAY_OF_WEEK_Vec_6    0.018524
10  DESTINATION_AIRPORT_Vec_10397    0.017106
11                 AIRLINE_Vec_B6    0.015842
12                 AIRLINE_Vec_F9    0.015490
13                    MONTH_Vec_7    0.015089
14    DESTINATION_AIRPORT_Vec_SLC    0.014688


In [34]:
from pyspark.ml.classification import LinearSVC

stages_svm = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_svm += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_svm = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_svm.append(assembler_svm)

svm = LinearSVC(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=20,
    regParam=0.1
)
stages_svm.append(svm)

pipeline_svm = Pipeline(stages=stages_svm)

In [35]:
model_svm = pipeline_svm.fit(train_data_weighted)
predictions_svm = model_svm.transform(test_data)

In [36]:
model_svm_path = "model/flight_svm_model"
model_svm.write().overwrite().save(model_svm_path)

In [37]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_svm)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_svm)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_svm)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_svm)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_svm)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9251
2. Accuracy:      0.9285
3. Precision:     0.9278
4. Recall:        0.9285
5. F1-Score:      0.9237


In [38]:
y_compare_svm = predictions_svm.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_svm["label"], y_compare_svm["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.93      0.99      0.96    941207
       Delay       0.92      0.66      0.77    204092

    accuracy                           0.93   1145299
   macro avg       0.92      0.82      0.86   1145299
weighted avg       0.93      0.93      0.92   1145299



In [39]:
svm_model = model_svm.stages[-1]
coefficients = svm_model.coefficients.toArray()

features_metadata = predictions_svm.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LINEAR SVM) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LINEAR SVM) ===
                     Feature_Name  Coefficient  Absolute_Importance
0        ORIGIN_AIRPORT_Vec_13964     1.164705             1.164705
1        ORIGIN_AIRPORT_Vec_14222     1.028976             1.028976
2          ORIGIN_AIRPORT_Vec_ADK     0.995720             0.995720
3          ORIGIN_AIRPORT_Vec_GST     0.982596             0.982596
4        ORIGIN_AIRPORT_Vec_15497     0.963609             0.963609
5   DESTINATION_AIRPORT_Vec_13964     0.952021             0.952021
6        ORIGIN_AIRPORT_Vec_10165     0.895629             0.895629
7          ORIGIN_AIRPORT_Vec_PPG     0.766168             0.766168
8   DESTINATION_AIRPORT_Vec_12016    -0.749576             0.749576
9        ORIGIN_AIRPORT_Vec_10154     0.729248             0.729248
10    DESTINATION_AIRPORT_Vec_GUM     0.713920             0.713920
11       ORIGIN_AIRPORT_Vec_13541     0.665345             0.665345
12  DESTINATION_AIRPORT_Vec_13502     0.665345             0.665

In [40]:
from pyspark.ml.classification import NaiveBayes

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DISTANCE", "DEPARTURE_DELAY"]  # bỏ DEP_TIME_MINUTES vì có thể âm

stages_nb = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    stages_nb.append(indexer)

assembler_inputs = [col + "_Index" for col in categorical_cols] + numeric_cols
assembler_nb = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_nb.append(assembler_nb)

nb = NaiveBayes(
    featuresCol="features",
    labelCol="label",
    smoothing=1.0,
    modelType="gaussian"  # dùng gaussian vì features là continuous
)
stages_nb.append(nb)

pipeline_nb = Pipeline(stages=stages_nb)

In [41]:
model_nb = pipeline_nb.fit(train_data)
predictions_nb = model_nb.transform(test_data)

In [42]:
model_nb_path = "model/flight_naive_bayes_model"
model_nb.write().overwrite().save(model_nb_path)

In [43]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_nb)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_nb)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_nb)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_nb)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_nb)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.5277
2. Accuracy:      0.9333
3. Precision:     0.9315
4. Recall:        0.9333
5. F1-Score:      0.9303


In [44]:
y_compare_nb = predictions_nb.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_nb["label"], y_compare_nb["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.94      0.98      0.96    941207
       Delay       0.89      0.71      0.79    204092

    accuracy                           0.93   1145299
   macro avg       0.92      0.85      0.88   1145299
weighted avg       0.93      0.93      0.93   1145299



In [45]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("=== TUNING LOGISTIC REGRESSION ===")

# Build pipeline không có LR ở cuối
stages_lr_tune = [s for s in stages if not isinstance(s, LogisticRegression)]

lr_tune = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
)
stages_lr_tune.append(lr_tune)
pipeline_lr_tune = Pipeline(stages=stages_lr_tune)

param_grid_lr = ParamGridBuilder() \
    .addGrid(lr_tune.maxIter, [20, 50]) \
    .addGrid(lr_tune.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr_tune.elasticNetParam, [0.0, 0.5]) \
    .build()

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

cv_lr = CrossValidator(
    estimator=pipeline_lr_tune,
    estimatorParamMaps=param_grid_lr,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

# Lấy 30% train_data_weighted để tuning
train_sample, _ = train_data_weighted.randomSplit([0.3, 0.7], seed=42)
print(f"Full train size:   {train_data_weighted.count()}")
print(f"Sample train size: {train_sample.count()}")

cv_model_lr = cv_lr.fit(train_sample)
best_model_lr = cv_model_lr.bestModel

print(f"Best maxIter:        {best_model_lr.stages[-1].getMaxIter()}")
print(f"Best regParam:       {best_model_lr.stages[-1].getRegParam()}")
print(f"Best elasticNet:     {best_model_lr.stages[-1].getElasticNetParam()}")

predictions_lr_tuned = best_model_lr.transform(test_data)

roc_auc = evaluator.evaluate(predictions_lr_tuned)
print(f"\nROC-AUC after tuning: {roc_auc:.4f}")

=== TUNING LOGISTIC REGRESSION ===
Full train size:   4583896
Sample train size: 1375933
Best maxIter:        20
Best regParam:       0.01
Best elasticNet:     0.5

ROC-AUC after tuning: 0.9305


In [46]:
print("=== RESULTS AFTER TUNING (LR) ===")

roc_auc = evaluator.evaluate(predictions_lr_tuned)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
print(f"2. Accuracy:      {acc_evaluator.evaluate(predictions_lr_tuned):.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
print(f"3. Precision:     {prec_evaluator.evaluate(predictions_lr_tuned):.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
print(f"4. Recall:        {rec_evaluator.evaluate(predictions_lr_tuned):.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
print(f"5. F1-Score:      {f1_evaluator.evaluate(predictions_lr_tuned):.4f}")

=== RESULTS AFTER TUNING (LR) ===
1. ROC-AUC Score: 0.9305
2. Accuracy:      0.9221
3. Precision:     0.9230
4. Recall:        0.9221
5. F1-Score:      0.9225


In [47]:
print("=== TUNING GBT ===")

stages_gbt_tune = [s for s in stages_gbt if not isinstance(s, GBTClassifier)]

gbt_tune = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
)
stages_gbt_tune.append(gbt_tune)
pipeline_gbt_tune = Pipeline(stages=stages_gbt_tune)

train_sample, _ = train_data_weighted.randomSplit([0.1, 0.9], seed=42)

param_grid_gbt = ParamGridBuilder() \
    .addGrid(gbt_tune.maxIter, [10, 20]) \
    .addGrid(gbt_tune.maxDepth, [4, 6]) \
    .build()

cv_gbt = CrossValidator(
    estimator=pipeline_gbt_tune,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

cv_model_gbt = cv_gbt.fit(train_sample)
best_model_gbt = cv_model_gbt.bestModel

print(f"Best maxIter:    {best_model_gbt.stages[-1].getMaxIter()}")
print(f"Best maxDepth:   {best_model_gbt.stages[-1].getMaxDepth()}")

predictions_gbt_tuned = best_model_gbt.transform(test_data)

roc_auc = evaluator.evaluate(predictions_gbt_tuned)
print(f"\nROC-AUC after tuning: {roc_auc:.4f}")

=== TUNING GBT ===
Best maxIter:    20
Best maxDepth:   6

ROC-AUC after tuning: 0.9350


In [48]:
print("=== RESULTS AFTER TUNING (GBT) ===")

roc_auc = evaluator.evaluate(predictions_gbt_tuned)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

print(f"2. Accuracy:      {acc_evaluator.evaluate(predictions_gbt_tuned):.4f}")
print(f"3. Precision:     {prec_evaluator.evaluate(predictions_gbt_tuned):.4f}")
print(f"4. Recall:        {rec_evaluator.evaluate(predictions_gbt_tuned):.4f}")
print(f"5. F1-Score:      {f1_evaluator.evaluate(predictions_gbt_tuned):.4f}")

=== RESULTS AFTER TUNING (GBT) ===
1. ROC-AUC Score: 0.9350
2. Accuracy:      0.9094
3. Precision:     0.9163
4. Recall:        0.9094
5. F1-Score:      0.9119


In [49]:
best_lr = cv_model_lr.bestModel.stages[-1]

lr_final = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=best_lr.getMaxIter(),
    regParam=best_lr.getRegParam(),
    elasticNetParam=best_lr.getElasticNetParam()
)

stages_lr_final = [s for s in stages if not isinstance(s, LogisticRegression)]
stages_lr_final.append(lr_final)

pipeline_lr_final = Pipeline(stages=stages_lr_final)
model_lr_final = pipeline_lr_final.fit(train_data_weighted)
predictions_lr_final = model_lr_final.transform(test_data)

In [50]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_lr_final)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_lr_final)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_lr_final)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_lr_final)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_lr_final)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9304
2. Accuracy:      0.9222
3. Precision:     0.9230
4. Recall:        0.9222
5. F1-Score:      0.9226


In [51]:
from sklearn.metrics import classification_report
y_compare = predictions_lr_final.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare["label"], y_compare["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.95      0.95    941207
       Delay       0.77      0.80      0.78    204092

    accuracy                           0.92   1145299
   macro avg       0.86      0.87      0.87   1145299
weighted avg       0.92      0.92      0.92   1145299



In [52]:
lr_model = model_lr_final.stages[-1]
coefficients = lr_model.coefficients.toArray()

features_metadata = predictions_lr_final.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===
                   Feature_Name  Coefficient  Absolute_Importance
0                AIRLINE_Vec_DL    -0.276751             0.276751
1                AIRLINE_Vec_NK     0.194472             0.194472
2                AIRLINE_Vec_WN    -0.183542             0.183542
3                  MONTH_Vec_10    -0.180745             0.180745
4                AIRLINE_Vec_F9     0.169018             0.169018
5   DESTINATION_AIRPORT_Vec_LGA     0.152520             0.152520
6   DESTINATION_AIRPORT_Vec_LAX     0.137294             0.137294
7                   MONTH_Vec_2     0.134387             0.134387
8                AIRLINE_Vec_UA    -0.126394             0.126394
9                   MONTH_Vec_9    -0.125163             0.125163
10  DESTINATION_AIRPORT_Vec_ORD     0.074408             0.074408
11       ORIGIN_AIRPORT_Vec_LGA     0.072567             0.072567
12              DEPARTURE_DELAY     0.069644             0.069644
13             

In [53]:
best_gbt = cv_model_gbt.bestModel.stages[-1]

gbt_final = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=best_gbt.getMaxIter(),
    maxDepth=best_gbt.getMaxDepth(),
    stepSize=best_gbt.getStepSize()
)

stages_gbt_final = [s for s in stages_gbt if not isinstance(s, GBTClassifier)]
stages_gbt_final.append(gbt_final)

pipeline_gbt_final = Pipeline(stages=stages_gbt_final)
model_gbt_final = pipeline_gbt_final.fit(train_data_weighted)
predictions_gbt_final = model_gbt_final.transform(test_data)

In [54]:
# ── OPTIMIZATION: Explain execution plan ───────────────────────
print("=== EXECUTION PLAN (GBT predictions) ===")
predictions_gbt_final.explain(extended=True)

=== EXECUTION PLAN (GBT predictions) ===
== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(prediction, UDF('rawPrediction) AS prediction#514399, None)]
+- Project [YEAR#100, MONTH#101, DAY#102, DAY_OF_WEEK#103, AIRLINE#104, FLIGHT_NUMBER#105, TAIL_NUMBER#106, ORIGIN_AIRPORT#107, DESTINATION_AIRPORT#108, SCHEDULED_DEPARTURE#257, DEPARTURE_TIME#258, DEPARTURE_DELAY#111, TAXI_OUT#112, WHEELS_OFF#113, SCHEDULED_TIME#114, ELAPSED_TIME#115, AIR_TIME#116, DISTANCE#117, WHEELS_ON#118, TAXI_IN#119, SCHEDULED_ARRIVAL#259, ARRIVAL_TIME#260, ARRIVAL_DELAY#122, DIVERTED#123, CANCELLED#124, ... 21 more fields]
   +- Project [YEAR#100, MONTH#101, DAY#102, DAY_OF_WEEK#103, AIRLINE#104, FLIGHT_NUMBER#105, TAIL_NUMBER#106, ORIGIN_AIRPORT#107, DESTINATION_AIRPORT#108, SCHEDULED_DEPARTURE#257, DEPARTURE_TIME#258, DEPARTURE_DELAY#111, TAXI_OUT#112, WHEELS_OFF#113, SCHEDULED_TIME#114, ELAPSED_TIME#115, AIR_TIME#116, DISTANCE#117, WHEELS_ON#118, TAXI_IN#119, SCHEDULED_ARRIVAL#259, ARRIVAL_TIME#26

In [55]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_gbt_final)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_gbt_final)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_gbt_final)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_gbt_final)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_gbt_final)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9355
2. Accuracy:      0.9119
3. Precision:     0.9177
4. Recall:        0.9119
5. F1-Score:      0.9141


In [56]:
y_compare_gbt = predictions_gbt.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_gbt["label"], y_compare_gbt["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.93      0.94    941207
       Delay       0.71      0.82      0.76    204092

    accuracy                           0.91   1145299
   macro avg       0.84      0.88      0.85   1145299
weighted avg       0.92      0.91      0.91   1145299



In [57]:
gbt_model = model_gbt_final.stages[-1]
feature_importances = gbt_model.featureImportances.toArray()

features_metadata = predictions_gbt_final.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (GBT) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (GBT) ===
                   Feature_Name  Importance
0               DEPARTURE_DELAY    0.912654
1                      DISTANCE    0.010828
2                AIRLINE_Vec_WN    0.008288
3                AIRLINE_Vec_UA    0.007570
4   DESTINATION_AIRPORT_Vec_ORD    0.004950
5                AIRLINE_Vec_DL    0.004599
6   DESTINATION_AIRPORT_Vec_LAX    0.004168
7                   MONTH_Vec_2    0.003786
8        ORIGIN_AIRPORT_Vec_LGA    0.003525
9   DESTINATION_AIRPORT_Vec_LGA    0.003337
10                  MONTH_Vec_1    0.003308
11             DEP_TIME_MINUTES    0.003073
12               AIRLINE_Vec_HA    0.002301
13               AIRLINE_Vec_US    0.002252
14       ORIGIN_AIRPORT_Vec_ATL    0.001847


In [58]:
# ── Unpersist after training ─────────────────────────────────────
flights.unpersist()
train_data.unpersist()
test_data.unpersist()
train_data_weighted.unpersist()
print("Unpersisted all cached data.")

Unpersisted all cached data.
